# 02 — Company Data Cleaning & Merging

**Owner**: Person B

Merge AGC member list (222 companies) and DOT prequalified contractors (57 companies)
into a unified company dataset. Deduplicate by company name.

### Input files
- `../data/processed/agc_members.csv` (from notebook 01)
- `../data/processed/dot_prequal.csv` (from notebook 01)

### Output files
- `../data/processed/companies_merged.csv`

In [1]:
import pandas as pd

agc = pd.read_csv('../data/processed/agc_members.csv')
dot = pd.read_csv('../data/processed/dot_prequal.csv')

print(f"AGC Members: {len(agc)}")
print(f"DOT Prequal: {len(dot)}")
print(f"\nAGC columns: {agc.columns.tolist()}")
print(f"DOT columns: {dot.columns.tolist()}")

AGC Members: 222
DOT Prequal: 57

AGC columns: ['type', 'member_name']
DOT columns: ['company_name', 'website']


In [2]:
# Standardize column names and merge
agc_clean = agc.rename(columns={'member_name': 'company_name'})
agc_clean['source'] = 'AGC'

dot_clean = dot.copy()
dot_clean['source'] = 'DOT'
dot_clean['type'] = 'DOT Prequalified'

# Merge
merged = pd.concat([agc_clean, dot_clean], ignore_index=True)
print(f"Before dedup: {len(merged)}")

# Deduplicate by normalized company name
merged['name_clean'] = merged['company_name'].str.strip().str.upper()
merged = merged.drop_duplicates(subset='name_clean', keep='first')
merged = merged.drop(columns='name_clean')

print(f"After dedup: {len(merged)}")
print(f"\nSource distribution:")
print(merged['source'].value_counts())
print(f"\nType distribution:")
print(merged['type'].value_counts())

# Save
merged.to_csv('../data/processed/companies_merged.csv', index=False)
print(f"\nSaved companies_merged.csv")

Before dedup: 279
After dedup: 272

Source distribution:
source
AGC    222
DOT     50
Name: count, dtype: int64

Type distribution:
type
Specialty Contractors    67
General Contractors      60
Service Providers        58
DOT Prequalified         50
Suppliers                35
Developer                 2
Name: count, dtype: int64

Saved companies_merged.csv
